# 11 为什么语言模型 loss 要按有效 token 归一化？

## 面试回答主线

语言模型的交叉熵应以有效预测 token 为分母，而不是以序列条数为分母。否则长文档和短问答对梯度的权重会被错误扭曲，padding 或截断策略也会改变目标函数。面试时要同时说明 ignore index、label shift、分布式 all-reduce 分子分母以及变长数据的 token budget。下面用六条客服对话的真实长度和每 token NLL，比较 sequence-average 与 token-average，并构造把 PAD 当真实 token 的失败。

**核心公式：** $L=-\frac{1}{\sum_i m_i}\sum_i\sum_t m_{it}\log p(y_{it}|x_{i,<t})$，其中 $m_{it}$ 是有效 label mask。不能用 $\frac1N\sum_i L_i$ 替代，除非每条序列有效 token 数完全相同。

下面按真实案例、基线、手写机制、结果表和失败修复组织回答；所有数据都是可复现的教学实验。


## 真实案例

场景是客服与账户安全系统中的六条脱敏离线事件。字段包含工单文本、有效 token 数和风险标签；它们模拟真实的数据结构，但样本极小，只用于观察公式和状态变化。


In [1]:
import math  # 导入数学函数以实现训练与掩码公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖的非教学弃用提示。
import torch  # 导入张量计算和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(29)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制小实验 CPU 线程数。
samples = [  # 构造六条脱敏客服对话作为真实语义样本。
    {'id': 'C01', 'text': '支付重复扣款，申请退款', 'tokens': 6, 'risk': 1},  # 资金风险工单。
    {'id': 'C02', 'text': '收不到登录验证码', 'tokens': 2, 'risk': 0},  # 登录支持工单。
    {'id': 'C03', 'text': '账户有陌生转账记录', 'tokens': 5, 'risk': 1},  # 账户安全工单。
    {'id': 'C04', 'text': '修改订单收货地址', 'tokens': 3, 'risk': 0},  # 售后咨询工单。
    {'id': 'C05', 'text': '银行卡盗刷需要冻结', 'tokens': 7, 'risk': 1},  # 高优先级安全工单。
    {'id': 'C06', 'text': '更正发票抬头信息', 'tokens': 4, 'risk': 0},  # 账单服务工单。
]  # 结束教学数据定义。
features = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [1.0, 1.0, 0.0], [0.0, 1.0, 1.0]])  # 构造三维可解释特征。
labels = torch.tensor([1, 0, 1, 0, 1, 0])  # 构造风险分类标签。
print('教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。')  # 声明数据边界。
for row in samples:  # 逐条展示真实语义输入。
    print(f"{row['id']} | token={row['tokens']} | risk={row['risk']} | {row['text']}")  # 输出样本字段。
print(f'特征形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状。


教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。
C01 | token=6 | risk=1 | 支付重复扣款，申请退款
C02 | token=2 | risk=0 | 收不到登录验证码
C03 | token=5 | risk=1 | 账户有陌生转账记录
C04 | token=3 | risk=0 | 修改订单收货地址
C05 | token=7 | risk=1 | 银行卡盗刷需要冻结
C06 | token=4 | risk=0 | 更正发票抬头信息
特征形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先在同一批六条事件上运行最简单方案。基线不是稻草人，它提供固定的输入、口径和可比较指标。


In [2]:
sequence_nll = [[0.12, 0.20, 0.18, 0.16], [0.50, 0.55], [0.08, 0.09, 0.10, 0.11, 0.12], [0.35, 0.32, 0.40], [0.18, 0.22, 0.20, 0.19, 0.21, 0.17], [0.60]]  # 记录六条客服回复每个有效 token 的负对数似然。
per_sequence = [sum(row) / len(row) for row in sequence_nll]  # 先对每条序列求均值模拟错误的 sequence-average。
baseline_metric = sum(per_sequence) / len(per_sequence)  # 再对六条序列平均得到基线 loss。
print(f'每条序列平均 NLL={ [round(value, 3) for value in per_sequence] }，sequence-average={baseline_metric:.4f}')  # 展示短回复被同等加权的基线。


每条序列平均 NLL=[0.165, 0.525, 0.1, 0.357, 0.195, 0.6]，sequence-average=0.3236


## 手写核心实现与中间量

代码保留关键分子分母、mask、梯度、参数组或重算路径，而不让 Trainer 或高层框架隐藏面试问题本身。


In [3]:
flat_nll = [value for row in sequence_nll for value in row]  # 展开所有有效 token 的损失。
valid_tokens = len(flat_nll)  # 统计真实参与目标函数的 token 数。
token_loss_sum = sum(flat_nll)  # 累加所有有效 token 的损失分子。
core_metric = token_loss_sum / valid_tokens  # 按有效 token 归一化交叉熵。
length_weights = [len(row) / valid_tokens for row in sequence_nll]  # 计算各对话在 token 目标中的真实权重。
print(f'有效 token={valid_tokens}，损失分子={token_loss_sum:.3f}，token-average={core_metric:.4f}')  # 输出分子分母与核心 loss。
print(f'六条对话的 token 权重={ [round(value, 3) for value in length_weights] }')  # 显示长短样本为何权重不同。


有效 token=21，损失分子=5.050，token-average=0.2405
六条对话的 token 权重=[0.19, 0.095, 0.238, 0.143, 0.286, 0.048]


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立同一口径的结果表。
for name, metric in comparison_rows:  # 逐行输出结果。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示可读数值对照。


Baseline | 指标=0.323611
核心机制     | 指标=0.240476


## 结果解读

基线和核心输出只在本受控案例中比较。生产训练必须跨数据并行 rank 同步 token 分子和分母；梯度累积、packing 和丢弃样本时也要保持计数一致。 观察结果时应关注中间量是否符合公式，而不是把六条样本上的数字宣传为线上收益。

## 失败案例

下一个单元故意破坏关键假设，并用实现修复证明该假设为何必要。


In [5]:
padded_width = 6  # 设定为了组 batch 而补齐到的固定长度。
padded_nll = [row + [0.70] * (padded_width - len(row)) for row in sequence_nll]  # 故意把 PAD 位置填成看似合理但不应计入的损失。
failure_metric = sum(sum(row) for row in padded_nll) / (padded_width * len(padded_nll))  # 错误地把 PAD 当真实 token 归一化。
fix_metric = token_loss_sum / valid_tokens  # 恢复按 mask 统计的正确 token loss。
print(f'失败：PAD 也参与 loss={failure_metric:.4f}；修复：只计有效 token={fix_metric:.4f}')  # 展示 ignore mask 的必要性。


失败：PAD 也参与 loss=0.4319；修复：只计有效 token=0.2405


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产训练必须跨数据并行 rank 同步 token 分子和分母；梯度累积、packing 和丢弃样本时也要保持计数一致。

**常见坑：** 把每条序列的均值再平均，或忘记将 PAD 的 label 设为 ignore index，会让短样本被过度放大。

**延伸追问：** SFT 中 prompt token 与 completion token 要不要都计 loss？如何在 packing 后统计每个 rank 的有效 token？

## 生产差距

本 Notebook 在 CPU/FP32 下处理 6 条离线事件，省略了真实 token packing、分布式同步、混合精度、checkpoint、隐私治理、监控告警和灰度回滚。生产版本必须替换为受审计的数据管道与系统级指标。


In [6]:
assert valid_tokens == 21  # 验证有效 token 计数来自六条原始回复。
assert abs(core_metric - fix_metric) < 1e-12  # 验证修复复用了同一 token 归一化定义。
assert abs(failure_metric - fix_metric) > 1e-3  # 验证把 PAD 计入目标会改变 loss。
assert 0.0 < core_metric < 1.0  # 验证教学 NLL 位于合理范围。
